# Deep Learning with PyTorch · Loss functions and optimisers

Macademia · For adults

> **Running in Colab:** Open this file at [colab.research.google.com](https://colab.research.google.com) with **File → Upload notebook**. For a GPU choose **Runtime → Change runtime type → T4 GPU**. Run the cells from top to bottom (Shift + Enter). PyTorch is preinstalled in Colab; to run on your own computer use `pip install torch torchvision`.

In [ ]:
import torch, random, numpy as np
torch.manual_seed(0); random.seed(0); np.random.seed(0)          # reproducibility
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| device:', device)

## 1. Loss functions

In [ ]:
from torch import nn
logits = torch.tensor([[2.0, 0.5, -1.0]]); target = torch.tensor([0])
ce = nn.CrossEntropyLoss()(logits, target)
manual = -torch.log_softmax(logits, dim=1)[0, 0]
print('CE', ce.item(), '| manual', manual.item())
print('double softmax (wrong):', nn.CrossEntropyLoss()(torch.softmax(logits, 1), target).item())
print('BCEWithLogits', nn.BCEWithLogitsLoss()(torch.tensor([1.2]), torch.tensor([1.0])).item())

## 2. Optimisers in a narrow valley
$f(x, y) = 0.04x^2 + y^2$

In [ ]:
import matplotlib.pyplot as plt
def run(make, steps=100):
    xy = torch.tensor([-8.0, 1.6], requires_grad=True); opt = make([xy]); path = [xy.detach().clone()]
    for _ in range(steps):
        opt.zero_grad(); f = 0.04 * xy[0] ** 2 + xy[1] ** 2; f.backward(); opt.step(); path.append(xy.detach().clone())
    return torch.stack(path)
opts = {'SGD': lambda p: torch.optim.SGD(p, lr=0.05), 'Momentum': lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9),
        'RMSprop': lambda p: torch.optim.RMSprop(p, lr=0.3), 'Adam': lambda p: torch.optim.Adam(p, lr=0.3)}
X, Y = np.meshgrid(np.linspace(-10, 10, 200), np.linspace(-3, 3, 100))
plt.contour(X, Y, 0.04 * X ** 2 + Y ** 2, levels=20, cmap='gray')
for name, make in opts.items():
    p = run(make); plt.plot(p[:, 0], p[:, 1], label=name)
plt.legend(); plt.title('Optimiser race'); plt.show()

## 3. Learning-rate sweep

In [ ]:
for lr in [0.001, 0.01, 0.1, 0.5, 1.0, 1.1]:
    p = run(lambda q: torch.optim.SGD(q, lr=lr))
    f = (0.04 * p[-1, 0] ** 2 + p[-1, 1] ** 2).item()
    print(f'lr={lr:<6} f={f:.3e}')                                   # diverges above 1.0

## 4. Scheduler

In [ ]:
w = torch.zeros(1, requires_grad=True)
opt = torch.optim.SGD([w], lr=0.1)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20)
lrs = []
for _ in range(20):
    opt.step(); sched.step(); lrs.append(opt.param_groups[0]['lr'])
plt.plot(lrs); plt.title('CosineAnnealingLR'); plt.show()

## Exercises
1. Run the same race on the Rosenbrock function (`(1-x)^2 + 10(y-x^2)^2`).
2. Set momentum to 0.5 and 0.99; compare the trajectories.
3. Plot the learning-rate curve of `OneCycleLR`.